In [0]:
# Gold — Inspect Sales Fact Source

sales = spark.table("bike_lakehouse.silver.crm_sales_details")

print("Sales rows:", sales.count())

display(
    sales.limit(20)
)

In [0]:
# Gold — Validate Sales Fact Grain

duplicate_grain = (
    sales
    .groupBy("order_number", "product_key")
    .count()
    .filter("count > 1")
)

print("Sales rows:", sales.count())
print("Duplicate order-line keys:", duplicate_grain.count())

display(
    duplicate_grain.limit(20)
)

In [0]:
# Gold — Validate Fact Dimension Relationships

dim_customers = spark.table("bike_lakehouse.gold.dim_customers")
dim_products = spark.table("bike_lakehouse.gold.dim_products")

customer_check = (
    sales
    .join(
        dim_customers,
        sales.customer_id == dim_customers.customer_id,
        "left"
    )
)

product_check = (
    sales.alias("s")
    .join(
        dim_products.alias("p"),
        dim_products.product_key.endswith(sales.product_key),
        "left"
    )
)

print(
    "Sales rows without customer match:",
    customer_check
    .filter(dim_customers.customer_id.isNull())
    .count()
)

print(
    "Sales rows without product match:",
    product_check
    .filter(dim_products.product_id.isNull())
    .count()
)

In [0]:
# Gold — Build Sales Fact

df_fact_sales = (
    sales
    .select(
        "order_number",
        "product_key",
        "customer_id",
        "order_date",
        "ship_date",
        "due_date",
        "sales_amount",
        "quantity",
        "unit_price"
    )
)

display(
    df_fact_sales.limit(20)
)

In [0]:
# Gold — Validate Sales Fact

print("Fact rows:", df_fact_sales.count())

print(
    "Duplicate order-line keys:",
    df_fact_sales
    .groupBy("order_number", "product_key")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "NULL order numbers:",
    df_fact_sales.filter("order_number IS NULL").count()
)

print(
    "NULL product keys:",
    df_fact_sales.filter("product_key IS NULL").count()
)

print(
    "NULL customer IDs:",
    df_fact_sales.filter("customer_id IS NULL").count()
)

print(
    "NULL order dates:",
    df_fact_sales.filter("order_date IS NULL").count()
)

print(
    "NULL quantities:",
    df_fact_sales.filter("quantity IS NULL").count()
)

In [0]:
# Gold — Write Sales Fact

df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.gold.fact_sales")

In [0]:
# Gold — Final Table Verification

gold_tables = [
    "bike_lakehouse.gold.dim_customers",
    "bike_lakehouse.gold.dim_products",
    "bike_lakehouse.gold.fact_sales"
]

for table_name in gold_tables:
    df = spark.table(table_name)
    print(f"{table_name}: {df.count()} rows")